<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/TIC-TAC-TOE_lektioner/Lab_2_Lektion_4_Din_Egen_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Tre i rad – Lektion 4: Din egen regelbaserade agent

**Målgrupp:** Gymnasiet, inga förkunskaper i programmering krävs  
**Tid:** ca 45 minuter  
**Mål:** Bygga en intelligent regelbaserad agent, lära sig if-satser för strategi, jämföra med slumpmässig agent

---

**Upphovspersoner:** LiU / Tekniksprånget  
**Licens:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.sv)

---

### 📚 Förutsättning
Den här lektionen bygger vidare på **Lektion 3**.  
Du ska känna till `find_empty_cells()` och `random_agent()` innan du fortsätter.

### 🗺️ Vad gör vi idag?
1. Lär oss vad en **regelbaserad AI** är
2. Bygger **Regel 1**: Vinn om möjligt 🏆
3. Bygger **Regel 2**: Blockera motspelaren 🛡️
4. Bygger **Regel 3**: Ta mitten ⭐
5. Kombinerar allt till en **smart agent** 🧠
6. Jämför smart agent mot slumpmässig agent 📊


In [ ]:
# ============================================================
# INSTÄLLNINGAR – Kör denna cell FÖRST!
# Innehåller all spellogik från tidigare lektioner.
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import random

BOARD_SIZE = 3   # Spelplanen är alltid 3x3

# ----------------------------------------------------------
# check_winner – kollar om någon spelare vann
# ----------------------------------------------------------
def check_winner(board):
    """Returnerar 1, 2 eller 0 (ingen vinnare ännu)."""
    for rad in range(BOARD_SIZE):
        if board[rad][0] != 0 and board[rad][0] == board[rad][1] == board[rad][2]:
            return board[rad][0]
    for kol in range(BOARD_SIZE):
        if board[0][kol] != 0 and board[0][kol] == board[1][kol] == board[2][kol]:
            return board[0][kol]
    if board[0][0] != 0 and board[0][0] == board[1][1] == board[2][2]:
        return board[0][0]
    if board[0][2] != 0 and board[0][2] == board[1][1] == board[2][0]:
        return board[0][2]
    return 0

# ----------------------------------------------------------
# make_move – gör ett drag om rutan är tom
# ----------------------------------------------------------
def make_move(board, row, col, player):
    """Returnerar True om draget lyckades, False annars."""
    if board[row][col] == 0:
        board[row][col] = player
        return True
    return False

# ----------------------------------------------------------
# visa_braede – skriver ut spelplanen som text
# ----------------------------------------------------------
def visa_braede(board):
    """Skriver ut spelplanen med symboler i terminalen."""
    symboler = {0: ".", 1: "X", 2: "O"}
    print("  0 1 2")
    for rad in range(BOARD_SIZE):
        rad_text = f"{rad} "
        for kol in range(BOARD_SIZE):
            rad_text += symboler[board[rad][kol]] + " "
        print(rad_text)
    print()

# ----------------------------------------------------------
# find_empty_cells – från Lektion 3
# ----------------------------------------------------------
def find_empty_cells(board):
    """Returnerar lista med tomma (rad, kol) positioner."""
    tomma = []
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:
                tomma.append((rad, kol))
    return tomma

# ----------------------------------------------------------
# random_agent – från Lektion 3
# ----------------------------------------------------------
def random_agent(board):
    """Väljer ett slumpmässigt drag bland de tomma cellerna."""
    tomma_celler = find_empty_cells(board)
    if tomma_celler:
        return random.choice(tomma_celler)
    return None

print("✅ Alla funktioner från tidigare lektioner är laddade!")


---
## 🧠 Del 1 – Vad är en regelbaserad AI?

### Den slumpmässiga agenten vs. den smarta agenten

I Lektion 3 byggde vi en **slumpmässig agent** – den väljer bara en ruta på måfå.

En **regelbaserad AI** är smartare: den följer **explicita regler** som vi skriver som kod.

> "Om X kan göra, gör Y"

Precis som en schackspelare som lär sig regler: *"Om du kan ta motståndarens drottning – ta den!"*

### Regler vi ska programmera

| # | Regel | Vad den gör |
|---|-------|------------|
| 1 | **Vinn om möjligt** 🏆 | Om det finns ett drag som vinner direkt – ta det! |
| 2 | **Blockera motspelaren** 🛡️ | Om motspelaren kan vinna nästa drag – stoppa dem! |
| 3 | **Ta mitten** ⭐ | Mitten (1,1) är strategiskt bäst om den är ledig |
| 4 | **Slumpmässigt** 🎲 | Om inget av ovanstående gäller – välj på måfå |

### Hur fungerar reglerna i kod?

```python
def smart_agent(board, player):
    # Regel 1: Kolla om vi kan vinna
    drag = hitta_vinnardrag(board, player)
    if drag:          # ← if-sats: gör regel 1 om den hittar något
        return drag

    # Regel 2: Kolla om vi måste blockera
    drag = hitta_blockdrag(board, player)
    if drag:          # ← if-sats: gör regel 2 om den hittar något
        return drag
    ...
```

**Ordningen spelar roll!** Regel 1 är viktigare än Regel 2 – vi vinner hellre än vi blockerar.


---
## 🏆 Del 2 – Regel 1: Vinn om möjligt

### Idén: "Prova och ångra"-metoden

Vi provar varje tomt fält:
1. Placera vår bricka där **tillfälligt**
2. Kolla med `check_winner()` om vi vinner
3. **Ångra** placeringen (sätt tillbaka 0)
4. Om vi vann → returnera det draget!

```
Spelplan:          Prova (0,1):      Prova (0,2):
X . .              X X .             X . X
X O .    →  →      X O .      →      X O .
. . .              . . .             . . .
                   ↓                 ↓
                 Vinner X!          Nej
                 Returnera (0,1)!
```

Vi behöver **inte** ångra om vi hittat ett vinnardrag – vi returnerar direkt!


In [ ]:
# ============================================================
# hitta_vinnardrag – Regel 1: Hitta ett drag som ger vinst
# ============================================================

def hitta_vinnardrag(board, player):
    """
    Kollar om spelaren kan vinna med nästa drag.

    Argument:
        board:  Spelplanen (2D-lista 3x3)
        player: Vilken spelare (1 eller 2)

    Returnerar:
        (rad, kol) om ett vinnardrag hittas
        None om inget vinnardrag finns
    """
    # Loopa igenom varje rad
    for rad in range(BOARD_SIZE):
        # Loopa igenom varje kolumn
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:   # Hittade en tom cell!

                # Steg 1: Placera vår bricka här tillfälligt
                board[rad][kol] = player

                # Steg 2: Kolla om vi vinner med det draget
                if check_winner(board) == player:
                    # Vi vinner! Ångra och returnera draget.
                    board[rad][kol] = 0   # Ångra placeringen
                    return (rad, kol)     # 🏆 Returnera vinnardraget!

                # Steg 3: Vi vann inte – ångra och prova nästa cell
                board[rad][kol] = 0   # Ångra placeringen

    # Ingen cell ger vinst
    return None  # Inget vinnardrag hittades


# ----------------------------------------------------------
# TEST – en spelplan där X (spelare 1) kan vinna
# ----------------------------------------------------------
test_vinna = [
    [1, 1, 0],   # ← X spelar (0,2) → vinner horisontellt!
    [2, 2, 0],
    [0, 0, 0]
]
print("Spelplan:")
visa_braede(test_vinna)
drag = hitta_vinnardrag(test_vinna, 1)   # Kan spelare 1 (X) vinna?
print(f"Spelare 1 kan vinna på: {drag}")
print(f"Förväntat: (0, 2)")
print()

# Kolla även för spelare 2
drag2 = hitta_vinnardrag(test_vinna, 2)
print(f"Spelare 2 kan vinna på: {drag2}")
print(f"Förväntat: (1, 2)")


---
## 🧪 Experiment – Regel 1 i olika situationer

Låt oss prova `hitta_vinnardrag()` på tre olika spelplaner:
- En där det **finns** ett vinnardrag
- En där det **inte finns** något vinnardrag
- En nästintill full plan


In [ ]:
# Experiment med hitta_vinnardrag på tre spelplaner

# Spelplan 1: Diagonal vinst möjlig för X
brade_a = [
    [1, 2, 0],
    [2, 1, 0],
    [0, 0, 0]   # ← X spelar (2,2) → vinner diagonalt!
]
print("Spelplan A:")
visa_braede(brade_a)
print(f"  Spelare 1 vinnardrag: {hitta_vinnardrag(brade_a, 1)}")
print(f"  Spelare 2 vinnardrag: {hitta_vinnardrag(brade_a, 2)}")
print()

# Spelplan 2: Ingen kan vinna direkt
brade_b = [
    [1, 2, 0],
    [0, 1, 2],
    [2, 0, 0]
]
print("Spelplan B (ingen direkt vinst):")
visa_braede(brade_b)
print(f"  Spelare 1 vinnardrag: {hitta_vinnardrag(brade_b, 1)}")
print(f"  Spelare 2 vinnardrag: {hitta_vinnardrag(brade_b, 2)}")
print()

# Spelplan 3: O nära att vinna
brade_c = [
    [1, 0, 0],
    [0, 2, 0],
    [0, 0, 2]   # ← O spelar (0,2) → vinner diagonalt!
]
print("Spelplan C:")
visa_braede(brade_c)
print(f"  Spelare 2 vinnardrag: {hitta_vinnardrag(brade_c, 2)}")


---
## 🛡️ Del 3 – Regel 2: Blockera motspelaren

### Idén: samma metod, men för motspelaren!

Blockera-regeln är nästan identisk med vinna-regeln – men vi kollar om **motspelaren** kan vinna.

Om motspelaren kan vinna → vi lägger vår bricka **på det draget** för att stoppa dem!

```
Spelplan:          O kan vinna på (2,0)!
X . O              → Blockera (2,0) nu!
X O .     
. . .              Spela X på (2,0):
                   X . O
                   X O .
                   X . .  ← Blockerat!
```

### Kodidé:
```python
def hitta_blockdrag(board, player):
    motstandare = 2 if player == 1 else 1   # Vem är motspelaren?
    return hitta_vinnardrag(board, motstandare)  # Kan motspelaren vinna?
```

Vi återanvänder `hitta_vinnardrag()` men frågar efter **motståndarens** vinnardrag! 🎯


In [ ]:
# ============================================================
# hitta_blockdrag – Regel 2: Blockera motståndarens vinnardrag
# ============================================================

def hitta_blockdrag(board, player):
    """
    Kollar om vi måste blockera motståndarens vinnardrag.

    Argument:
        board:  Spelplanen (2D-lista 3x3)
        player: Vilken spelare VI är (1 eller 2)

    Returnerar:
        (rad, kol) om ett blockdrag behövs
        None om motspelaren inte kan vinna direkt
    """
    # Ta reda på vem motspelaren är
    # Är vi spelare 1? Då är motspelaren spelare 2, och tvärtom.
    motstandare = 2 if player == 1 else 1

    # Återanvänder hitta_vinnardrag – men för MOTspelaren!
    # Om motspelaren kan vinna → blockera det draget
    return hitta_vinnardrag(board, motstandare)


# ----------------------------------------------------------
# TEST – en spelplan där O (spelare 2) hotar att vinna
# ----------------------------------------------------------
test_blockera = [
    [1, 0, 0],
    [0, 1, 0],
    [2, 2, 0]   # ← O spelar (2,2) → vinner horisontellt!
]
print("Spelplan (O hotar att vinna):")
visa_braede(test_blockera)
blockdrag = hitta_blockdrag(test_blockera, 1)   # Vi är spelare 1
print(f"Spelare 1 bör blockera på: {blockdrag}")
print(f"Förväntat: (2, 2)")
print()

# Kontroll: Finns det ett vinnardrag för spelare 1 också?
vinnardrag = hitta_vinnardrag(test_blockera, 1)
print(f"Spelare 1 eget vinnardrag: {vinnardrag}")
print("→ Regel 1 gäller FÖRE Regel 2, så vi kontrollerar vinnardrag först!")


---
## 🧪 Experiment – När Regel 2 räddar spelet

Prova spelplanen nedan. O är ett steg från att vinna – X måste blockera!


In [ ]:
# Experiment: Regel 2 räddar spelet

brade_hot = [
    [2, 0, 0],
    [0, 2, 0],   # O håller på att göra en diagonal!
    [0, 0, 0]    # ← O spelar (2,2) → vinner!
]
print("Spelplan – O hotar diagonalen:")
visa_braede(brade_hot)

# Regel 1: Kan X vinna?
regel1 = hitta_vinnardrag(brade_hot, 1)
print(f"Regel 1 (X vinnardrag): {regel1}")

# Regel 2: Måste X blockera?
regel2 = hitta_blockdrag(brade_hot, 1)
print(f"Regel 2 (blockdrag): {regel2}")

print()
if regel1:
    print("✅ X väljer att VINNA med Regel 1!")
elif regel2:
    print("🛡️ X väljer att BLOCKERA med Regel 2!")
else:
    print("🎲 Ingen akut situation – välj fritt.")


---
## ⭐ Del 4 – Regel 3: Ta mitten

### Varför är mitten viktig?

Mittcellen **(rad 1, kolumn 1)** ingår i **4 av 8 möjliga vinstlinjer**:
- Horisontell mittrad
- Vertikal mittkolumn
- Diagonal (vänster-höger)
- Diagonal (höger-vänster)

Jämför med ett **hörn** (2 vinstlinjer) eller en **kantcell** (bara 3 vinstlinjer).

Att ta mitten tidigt ger störst möjlighet att skapa hot åt flera håll!

```
Vinstlinjer som passerar mitten:
→ → →    (horisontell rad 1)
↓ ↓ ↓    (vertikal kol 1)
↘ ↘ ↘    (diagonal)
↙ ↙ ↙    (anti-diagonal)
```


In [ ]:
# ============================================================
# ta_mitten – Regel 3: Ta mittcellen om den är ledig
# ============================================================

def ta_mitten(board):
    """
    Tar mittcellen om den är ledig.

    Argument:
        board: Spelplanen (2D-lista 3x3)

    Returnerar:
        (1, 1) om mitten är tom
        None om mitten redan är tagen
    """
    # Mittcellen är alltid vid rad 1, kolumn 1
    if board[1][1] == 0:   # 0 = tom cell
        return (1, 1)      # ← Returnera mittenpositionen

    return None   # Mitten är redan tagen


# ----------------------------------------------------------
# TEST
# ----------------------------------------------------------
brade_tom = [[0]*3 for _ in range(3)]   # Helt tom spelplan
print("Tom spelplan:")
visa_braede(brade_tom)
print(f"ta_mitten returnerar: {ta_mitten(brade_tom)}")
print(f"Förväntat: (1, 1)")
print()

brade_mitten_tagen = [[0]*3 for _ in range(3)]
brade_mitten_tagen[1][1] = 2   # O har tagit mitten
print("Spelplan – O har mitten:")
visa_braede(brade_mitten_tagen)
print(f"ta_mitten returnerar: {ta_mitten(brade_mitten_tagen)}")
print(f"Förväntat: None")


---
## 🧠 Del 5 – Den smarta agenten: `smart_agent()`

Nu sätter vi ihop alla tre reglerna till en komplett agent!

### Prioritetsordningen:
```
Regel 1: Kan jag VINNA?           ← Viktigast!
    ↓ (Nej)
Regel 2: Måste jag BLOCKERA?      ← Näst viktigast
    ↓ (Nej)
Regel 3: Ta MITTEN om ledig?      ← Strategiskt
    ↓ (Nej)
Fallback: SLUMPMÄSSIGT drag       ← Alltid ett val
```

**Varför den ordningen?** 
- Det finns ingen mening att blockera om vi kan vinna direkt!
- Det finns ingen mening att ta mitten om vi riskerar att förlora!

### Flödesschema
```
smart_agent anropas
       ↓
  Kan vi vinna? → JA → Returnera vinnardrag
       ↓ NEJ
  Måste vi blockera? → JA → Returnera blockdrag
       ↓ NEJ
  Är mitten ledig? → JA → Returnera (1,1)
       ↓ NEJ
  Välj slumpmässigt → Returnera random
```


In [ ]:
# ============================================================
# smart_agent – Intelligent agent med tre regler
# ============================================================

def smart_agent(board, player):
    """
    Intelligent agent som följer tre regler i prioritetsordning.

    Argument:
        board:  Spelplanen (2D-lista 3x3)
        player: Vilken spelare vi är (1 eller 2)

    Returnerar:
        (rad, kol) – det valda draget
        None om inga drag är möjliga
    """
    # ----------------------------------------------------------
    # Regel 1: Vinn om möjligt – högsta prioritet!
    # ----------------------------------------------------------
    drag = hitta_vinnardrag(board, player)
    if drag:
        print(f"  🏆 Regel 1: Vinner på {drag}")
        return drag

    # ----------------------------------------------------------
    # Regel 2: Blockera motståndarens vinnardrag
    # ----------------------------------------------------------
    drag = hitta_blockdrag(board, player)
    if drag:
        print(f"  🛡️ Regel 2: Blockerar på {drag}")
        return drag

    # ----------------------------------------------------------
    # Regel 3: Ta mitten om den är ledig
    # ----------------------------------------------------------
    drag = ta_mitten(board)
    if drag:
        print(f"  ⭐ Regel 3: Tar mitten {drag}")
        return drag

    # ----------------------------------------------------------
    # Fallback: Slumpmässigt drag om inget annat passar
    # ----------------------------------------------------------
    tomma = find_empty_cells(board)
    if tomma:
        drag = random.choice(tomma)
        print(f"  🎲 Fallback: Slumpmässigt {drag}")
        return drag

    return None   # Spelplanen är full


# ----------------------------------------------------------
# TEST – Prova smart_agent på tre situationer
# ----------------------------------------------------------
print("=== Situation 1: Spelare 1 kan vinna ===")
brade_t1 = [
    [1, 1, 0],   # ← Spelare 1 kan vinna på (0,2)
    [2, 2, 0],   # ← Spelare 2 hotar (1,2) – men vi vinner först!
    [0, 0, 0]
]
visa_braede(brade_t1)
drag = smart_agent(brade_t1, 1)
print()

print("=== Situation 2: Måste blockera ===")
brade_t2 = [
    [1, 0, 0],
    [2, 2, 0],   # ← Spelare 2 vinner på (1,2)!
    [0, 0, 0]
]
visa_braede(brade_t2)
drag = smart_agent(brade_t2, 1)
print()

print("=== Situation 3: Tom spelplan – ta mitten ===")
brade_t3 = [[0]*3 for _ in range(3)]
visa_braede(brade_t3)
drag = smart_agent(brade_t3, 1)


---
## 🎮 Del 6 – Spela mot den smarta agenten!

Nu bygger vi det interaktiva spelet – men den här gången spelar du mot `smart_agent()`!

Agenten:
- Vinner direkt om den kan 🏆
- Blockerar dig om du är nära att vinna 🛡️
- Tar mitten om den är ledig ⭐

**Tips:** Kan du hitta en strategi för att vinna mot den?  
*(Hint: Prova att skapa **dubbelhot** – två sätt att vinna på samma gång!)*


In [ ]:
# ============================================================
# Interaktivt spel: Människa mot smart agent
# ============================================================

def starta_spel_mot_smart():
    """Startar ett interaktivt spel mot den smarta agenten."""

    brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    nuvarande_spelare = 1
    spelet_slut = False
    symboler = {1: "❌", 2: "⭕"}

    # Widgets
    knappar = {}
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            btn = widgets.Button(
                description=" ",
                layout=widgets.Layout(width="80px", height="80px"),
                style={"font_size": "28px"}
            )
            knappar[(rad, kol)] = btn

    status_etikett = widgets.Label(
        value="🎮 Ditt drag! Du är ❌",
        style={"font_size": "16px"}
    )
    info_etikett = widgets.Label(
        value="",
        style={"font_size": "13px"}
    )
    nytt_spel_knapp = widgets.Button(
        description="🔄 Nytt spel",
        button_style="info",
        layout=widgets.Layout(width="150px", height="40px")
    )

    def uppdatera_knappar():
        for (r, k), btn in knappar.items():
            val = brade[r][k]
            if val == 0:
                btn.description = " "
                btn.disabled = False
                btn.button_style = ""
            elif val == 1:
                btn.description = "❌"
                btn.disabled = True
                btn.button_style = "danger"
            else:
                btn.description = "⭕"
                btn.disabled = True
                btn.button_style = "success"

    def dator_drag():
        nonlocal nuvarande_spelare, spelet_slut
        # Smart agent – skriver ut vilken regel den använder
        import io, sys
        gammal_stdout = sys.stdout
        sys.stdout = io.StringIO()
        drag = smart_agent(brade, 2)
        regel_text = sys.stdout.getvalue().strip()
        sys.stdout = gammal_stdout

        info_etikett.value = f"🤖 {regel_text}" if regel_text else ""

        if drag:
            rad, kol = drag
            make_move(brade, rad, kol, 2)
            uppdatera_knappar()
            vinnare = check_winner(brade)
            if vinnare:
                status_etikett.value = f"🤖 Smarta agenten vinner! {symboler[vinnare]}"
                spelet_slut = True
                for btn in knappar.values():
                    btn.disabled = True
            elif not find_empty_cells(brade):
                status_etikett.value = "🤝 Oavgjort! Ingen vann."
                spelet_slut = True
            else:
                nuvarande_spelare = 1
                status_etikett.value = "🎮 Din tur! Du är ❌"

    def klick(btn_info, rad, kol):
        nonlocal nuvarande_spelare, spelet_slut
        if spelet_slut or nuvarande_spelare != 1:
            return
        if make_move(brade, rad, kol, 1):
            uppdatera_knappar()
            vinnare = check_winner(brade)
            if vinnare:
                status_etikett.value = f"🎉 Du vinner! {symboler[vinnare]}"
                info_etikett.value = "🥳 Imponerande! Du lyckades slå den smarta agenten!"
                spelet_slut = True
                for btn in knappar.values():
                    btn.disabled = True
            elif not find_empty_cells(brade):
                status_etikett.value = "🤝 Oavgjort!"
                spelet_slut = True
            else:
                nuvarande_spelare = 2
                status_etikett.value = "🤖 Agenten tänker..."
                dator_drag()

    for (r, k), btn in knappar.items():
        btn.on_click(lambda b, rad=r, kol=k: klick(b, rad, kol))

    def nytt_spel(b):
        nonlocal brade, nuvarande_spelare, spelet_slut
        brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
        nuvarande_spelare = 1
        spelet_slut = False
        uppdatera_knappar()
        status_etikett.value = "🎮 Ditt drag! Du är ❌"
        info_etikett.value = ""

    nytt_spel_knapp.on_click(nytt_spel)

    rader_layout = []
    for rad in range(BOARD_SIZE):
        rad_knappar = [knappar[(rad, kol)] for kol in range(BOARD_SIZE)]
        rader_layout.append(widgets.HBox(rad_knappar))

    spelplan_box = widgets.VBox(rader_layout)
    hela_spelet = widgets.VBox([
        widgets.Label("🎮 Tre i rad – Du (❌) mot Smart Agent (⭕)"),
        spelplan_box,
        status_etikett,
        info_etikett,
        nytt_spel_knapp
    ])
    display(hela_spelet)

starta_spel_mot_smart()


---
## 📊 Del 7 – Jämförelse: Smart agent mot slumpmässig agent

Nu kör vi 1 000 automatiska spel: **smart_agent** mot **random_agent**.

**Vad tror du händer?**  
- Vinner smart agenten alltid?
- Hur ofta blir det oavgjort?
- Spelar det roll om smart agenten börjar eller inte?


In [ ]:
# ============================================================
# Simulering: Smart agent mot slumpmässig agent
# ============================================================

import sys, io

def simulera_smart_vs_random(antal_spel, smart_spelar_som=1):
    """
    Simulerar spel mellan smart agent och slumpmässig agent.

    Argument:
        antal_spel:       Antal spel att simulera
        smart_spelar_som: 1 om smart agent börjar, 2 annars
    """
    resultat = {1: 0, 2: 0, "oavgjort": 0}

    for _ in range(antal_spel):
        brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
        nuvarande = 1

        while True:
            # Välj agent beroende på vems tur det är
            if nuvarande == smart_spelar_som:
                # Stäng av utskrifter från smart_agent
                sys.stdout = io.StringIO()
                drag = smart_agent(brade, nuvarande)
                sys.stdout = sys.__stdout__
            else:
                drag = random_agent(brade)

            if drag is None:
                resultat["oavgjort"] += 1
                break

            rad, kol = drag
            make_move(brade, rad, kol, nuvarande)

            vinnare = check_winner(brade)
            if vinnare:
                resultat[vinnare] += 1
                break

            if not find_empty_cells(brade):
                resultat["oavgjort"] += 1
                break

            nuvarande = 2 if nuvarande == 1 else 1

    return resultat


antal = 1000

# Smart som spelare 1 (börjar)
print(f"🎮 Simulering 1: Smart (❌) börjar mot Slumpmässig (⭕)")
stats1 = simulera_smart_vs_random(antal, smart_spelar_som=1)
print(f"  ❌ Smart vann:       {stats1[1]:4d} ({stats1[1]/antal*100:.1f}%)")
print(f"  ⭕ Slumpmässig vann: {stats1[2]:4d} ({stats1[2]/antal*100:.1f}%)")
print(f"  🤝 Oavgjort:         {stats1['oavgjort']:4d} ({stats1['oavgjort']/antal*100:.1f}%)")
print()

# Smart som spelare 2 (börjar inte)
print(f"🎮 Simulering 2: Slumpmässig (❌) börjar mot Smart (⭕)")
stats2 = simulera_smart_vs_random(antal, smart_spelar_som=2)
print(f"  ❌ Slumpmässig vann: {stats2[1]:4d} ({stats2[1]/antal*100:.1f}%)")
print(f"  ⭕ Smart vann:       {stats2[2]:4d} ({stats2[2]/antal*100:.1f}%)")
print(f"  🤝 Oavgjort:         {stats2['oavgjort']:4d} ({stats2['oavgjort']/antal*100:.1f}%)")
print()
print("💡 Jämförelse med Lektion 3 (slumpmässig vs slumpmässig): ~50% vardera!")


---
## ✏️ Övningar

Svara på frågorna eller kör kodcellerna för att prova dina kunskaper.


### 📝 Övning 1 – Spåra hitta_vinnardrag() steg för steg

Spelplan:
```
X . .
O X .
. . .
```
(Spelare 1 = X, Spelare 2 = O)

**a)** Gå igenom `hitta_vinnardrag(board, 1)` manuellt.  
Vilka tomma celler testas? För vilken cell "vinner" X?

*Skriv ditt svar här:*  
Svar: ...

**b)** Kontrollera med koden nedan.


In [ ]:
# Övning 1b
brade_ex1 = [
    [1, 0, 0],
    [2, 1, 0],
    [0, 0, 0]
]
print("Spelplan:")
visa_braede(brade_ex1)
drag = hitta_vinnardrag(brade_ex1, 1)
print(f"Spelare 1 vinnardrag: {drag}")


### 📝 Övning 2 – Vad händer om vi tar bort Regel 2?

**a)** Modifiera `smart_agent()` (i cellen nedan) så att den **hoppar över Regel 2**.  
Spela mot den – är det lättare att vinna nu?

**b)** I vilken situation blir skillnaden störst?

*Skriv ditt svar här:*  
Svar: ...


In [ ]:
# Övning 2 – Smart agent utan Regel 2

def smart_agent_utan_regel2(board, player):
    """Smart agent med Regel 1 och 3, men utan Regel 2 (blockering)."""
    # Regel 1: Vinn om möjligt
    drag = hitta_vinnardrag(board, player)
    if drag:
        return drag

    # TODO: Regel 2 är borttagen!

    # Regel 3: Ta mitten
    drag = ta_mitten(board)
    if drag:
        return drag

    # Fallback
    tomma = find_empty_cells(board)
    return random.choice(tomma) if tomma else None


# Test: Situation där blockering är viktig
brade_test = [
    [1, 0, 0],
    [2, 2, 0],   # O kan vinna på (1,2)!
    [0, 0, 0]
]
print("Spelplan – O hotar att vinna:")
visa_braede(brade_test)
print(f"Smart agent (med Regel 2): {smart_agent(brade_test, 1)}")

import sys, io
sys.stdout = io.StringIO()
drag_utan = smart_agent_utan_regel2(brade_test, 1)
sys.stdout = sys.__stdout__
print(f"Agent utan Regel 2:        {drag_utan}")


### 📝 Övning 3 – Lägg till Regel 4: Föredra hörn

Hörnen (0,0), (0,2), (2,0), (2,2) är strategiskt starka – de ingår i 2 vinstlinjer var.

**Uppgift:** Skapa funktionen `ta_horn(board)` som:
1. Kollar om något hörn är ledigt
2. Returnerar det första lediga hörnet
3. Returnerar `None` om alla hörn är tagna

Lägg sedan till funktionen i en ny agent `smart_agent_med_horn()`.


In [ ]:
# Övning 3 – Lägg till hörn-regel

def ta_horn(board):
    """
    Tar ett ledigt hörn (om något hörn är ledig).
    Hörnen är: (0,0), (0,2), (2,0), (2,2)
    """
    horn = [(0, 0), (0, 2), (2, 0), (2, 2)]

    for (rad, kol) in horn:
        if board[rad][kol] == 0:   # Hörnet är ledigt!
            return (rad, kol)

    return None   # Alla hörn är tagna


def smart_agent_med_horn(board, player):
    """Smart agent med fyra regler: Vinna, Blockera, Mitten, Hörn."""
    import sys, io

    # Regel 1: Vinn om möjligt
    drag = hitta_vinnardrag(board, player)
    if drag:
        return drag

    # Regel 2: Blockera
    drag = hitta_blockdrag(board, player)
    if drag:
        return drag

    # Regel 3: Ta mitten
    drag = ta_mitten(board)
    if drag:
        return drag

    # Regel 4: Ta ett hörn
    drag = ta_horn(board)
    if drag:
        return drag

    # Fallback
    tomma = find_empty_cells(board)
    return random.choice(tomma) if tomma else None


# Test
brade_horn = [
    [0, 0, 0],
    [0, 1, 0],   # Mitten tagen
    [0, 0, 0]
]
print("Spelplan – Mitten tagen, hörn lediga:")
visa_braede(brade_horn)
drag = smart_agent_med_horn(brade_horn, 2)
print(f"Agent med Hörn-regel väljer: {drag}")
print("Förväntat: ett hörn (0,0), (0,2), (2,0) eller (2,2)")


### 📝 Övning 4 – Vilken regel är viktigast?

Kör simuleringen nedan för att jämföra tre versioner:
1. Smart agent med **alla regler**
2. Smart agent **utan Regel 2** (ingen blockering)
3. Smart agent **utan Regel 3** (ingen mitten-preferens)

**Fundera:** Vilken regel gör störst skillnad mot slumpmässig agent?

*Skriv ditt svar här:*  
Svar: ...


In [ ]:
# Övning 4 – Jämförelse av regelversioner

def smart_utan_r3(board, player):
    """Smart agent utan Regel 3 (ingen mitten-preferens)."""
    drag = hitta_vinnardrag(board, player)
    if drag: return drag
    drag = hitta_blockdrag(board, player)
    if drag: return drag
    tomma = find_empty_cells(board)
    return random.choice(tomma) if tomma else None

def körbatch(agent_func, antal=500):
    """Kör agenten mot random_agent och returnerar vinstprocent."""
    vinster = 0
    for _ in range(antal):
        brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
        spelare = 1
        while True:
            if spelare == 1:
                import sys, io
                sys.stdout = io.StringIO()
                drag = agent_func(brade, 1)
                sys.stdout = sys.__stdout__
            else:
                drag = random_agent(brade)
            if not drag: break
            make_move(brade, drag[0], drag[1], spelare)
            v = check_winner(brade)
            if v == 1: vinster += 1; break
            if v == 2: break
            if not find_empty_cells(brade): break
            spelare = 2 if spelare == 1 else 1
    return vinster / antal * 100

print("Jämförelse – smart agent mot slumpmässig (500 spel per test):")
print()

pct_full = körbatch(smart_agent)
print(f"  ✅ Alla regler:            {pct_full:.1f}% vinster")

pct_no_r2 = körbatch(smart_agent_utan_regel2)
print(f"  ❌ Utan Regel 2 (blockera): {pct_no_r2:.1f}% vinster")

pct_no_r3 = körbatch(smart_utan_r3)
print(f"  ❌ Utan Regel 3 (mitten):   {pct_no_r3:.1f}% vinster")


### 📝 Övning 5 – Bygg din egen Regel 5!

Nu är det din tur att uppfinna en ny regel!

**Idéförslag:**
- `ta_kantcell(board)` – Ta en kantcell (0,1), (1,0), (1,2) eller (2,1) om ledig
- `föredra_hörn_mitt_emot(board, player)` – Ta ett hörn diagonalt mitt emot motståndarens hörn
- `skapa_dubbelhot(board, player)` – Hitta ett drag som skapar två vinstmöjligheter

Välj en idé, implementera den, och lägg till den i din nya `super_agent()`!


In [ ]:
# Övning 5 – Din egen Regel 5!

def min_regel_5(board, player):
    """
    Min egna regel – skriv din lösning här!

    Tips: Returnera (rad, kol) om din regel passar, annars None.
    """
    # TODO: Skriv din kod här!
    return None   # Byt ut detta mot din regel


def super_agent(board, player):
    """
    Super-agent med fem regler – du väljer Regel 5!
    """
    # Regel 1: Vinn om möjligt
    drag = hitta_vinnardrag(board, player)
    if drag: return drag

    # Regel 2: Blockera
    drag = hitta_blockdrag(board, player)
    if drag: return drag

    # Regel 3: Ta mitten
    drag = ta_mitten(board)
    if drag: return drag

    # Regel 4: Ta ett hörn
    drag = ta_horn(board)
    if drag: return drag

    # Regel 5: Din egen regel!
    drag = min_regel_5(board, player)
    if drag: return drag

    # Fallback
    tomma = find_empty_cells(board)
    return random.choice(tomma) if tomma else None


print("✅ super_agent är redo!")
print("Skriv din Regel 5 ovan och testa agenten mot random_agent!")


### 📝 Övning 6 – Spåra smart_agent() på en spelplan

Spelplan:
```
O . X
. O .
X . .
```
Det är X:s tur (spelare 1).

**a)** Kan X vinna direkt? Vilken cell? (Regel 1)

*Svar: ...*

**b)** Kan O vinna nästa drag om X inte blockerar? Vilken cell? (Regel 2)

*Svar: ...*

**c)** Vad väljer smart_agent? Kontrollera med koden nedan.


In [ ]:
# Övning 6 – Spåra smart_agent

brade_ex6 = [
    [2, 0, 1],
    [0, 2, 0],
    [1, 0, 0]
]
print("Spelplan:")
visa_braede(brade_ex6)
drag = smart_agent(brade_ex6, 1)
print(f"\nsmart_agent väljer: {drag}")


---
## 🧪 Quiz – Testa dina kunskaper!


In [ ]:
# Quiz 1: Vad returnerar hitta_vinnardrag() om det INTE finns ett vinnardrag?
svar_1 = "C"  # A="0", B="False", C="None", D="[]"
if svar_1 == "C":
    print("✅ Rätt! Funktionen returnerar None om inget vinnardrag hittas.")
else:
    print("❌ Fel! Titta på sista raden i hitta_vinnardrag() – vad returneras?")


In [ ]:
# Quiz 2: Varför ångrar vi draget i hitta_vinnardrag() (board[rad][kol] = 0)?
svar_2 = "B"
# A = "För att spara minne"
# B = "För att inte ändra den riktiga spelplanen – vi bara testar"
# C = "För att koden kraschar annars"
# D = "Det är ett misstag, vi behöver inte det"
if svar_2 == "B":
    print("✅ Rätt! Vi provar draget tillfälligt – men den riktiga spelplanen ska inte ändras.")
else:
    print("❌ Fel! Tänk på: vad händer med spelplanen om vi INTE ångrar?")


In [ ]:
# Quiz 3: Vilken regel har HÖGST prioritet i smart_agent()?
svar_3 = "A"
# A = "Regel 1: Vinn om möjligt"
# B = "Regel 2: Blockera"
# C = "Regel 3: Ta mitten"
# D = "Fallback: slumpmässigt"
if svar_3 == "A":
    print("✅ Rätt! Att vinna är alltid bättre än att blockera!")
else:
    print("❌ Fel! Titta på ordningen i smart_agent() – vilken regel kontrolleras FÖRST?")


In [ ]:
# Quiz 4: Vad returnerar hitta_blockdrag(board, 1)?
svar_4 = "B"
# A = "Det bästa draget för spelare 1"
# B = "Det drag som blockerar spelare 2:s vinnardrag"
# C = "Det sämsta draget för spelare 1"
# D = "Alltid (1, 1)"
if svar_4 == "B":
    print("✅ Rätt! hitta_blockdrag anropar hitta_vinnardrag för MOTspelaren.")
else:
    print("❌ Fel! Titta på koden: vems vinnardrag kollar hitta_blockdrag()?")


In [ ]:
# Quiz 5: Varför är mitten (1,1) strategiskt bäst?
svar_5 = "C"
# A = "Datorn väljer alltid mitten"
# B = "Det är lättast att komma åt med musen"
# C = "Mitten ingår i flest vinstlinjer (4 av 8)"
# D = "Det är regler i brädspelet att börja i mitten"
if svar_5 == "C":
    print("✅ Rätt! Mitten ingår i 4 av 8 möjliga vinstlinjer – hörnen bara 2.")
else:
    print("❌ Fel! Tänk på hur många vinstlinjer som passerar mittcellen.")


In [ ]:
# Quiz 6: Vad är en "regelbaserad AI"?
svar_6 = "A"
# A = "En AI som följer explicita om-då-regler skrivna av en programmerare"
# B = "En AI som lär sig av många spel automatiskt"
# C = "En AI som alltid vinner"
# D = "En AI som använder slump för att vara oförutsägbar"
if svar_6 == "A":
    print("✅ Rätt! Regelbaserad AI följer handskrivna if-satser – till skillnad från maskininlärning.")
else:
    print("❌ Fel! Tänk på vad 'regelbaserad' (eng. rule-based) betyder.")


---
## 🎓 Sammanfattning – Lektion 4

### Vad lärde vi oss?

✅ **Regelbaserad AI** – en agent som följer explicita om-då-regler  
✅ **hitta_vinnardrag()** – "prova och ångra"-metod för att hitta vinnardrag  
✅ **hitta_blockdrag()** – återanvänder vinna-logiken men för motspelaren  
✅ **ta_mitten()** – prioriterar strategiskt viktiga celler  
✅ **smart_agent()** – kombinerar alla regler i prioritetsordning  
✅ **Jämförelse** – smart agent vinner mot slumpmässig i ~80–95% av fallen  

### Nya funktioner du kan:
| Funktion | Vad den gör |
|----------|------------|
| `hitta_vinnardrag(board, player)` | Returnerar ett vinnardrag eller None |
| `hitta_blockdrag(board, player)` | Returnerar ett blockdrag eller None |
| `ta_mitten(board)` | Returnerar (1,1) om ledig, annars None |
| `smart_agent(board, player)` | Kombinerar alla tre regler |

### 🧠 Nyckelinsikt: Prioritetsordning
```
1. VINN    →  2. BLOCKERA  →  3. MITTEN  →  4. SLUMPMÄSSIGT
(Viktigast)                              (Fallback)
```

### 🚀 Vad händer härnäst?

Nu har du en regelbaserad AI – men den är inte **perfekt**.  
Det finns situationer där en klokare spelare kan vinna.

Nästa steg:
- 🌳 **Minimax-algoritmen** – den perfekta AI:n som räknar ut ALLA möjliga framtida drag
- �� **Maskininlärning** – en AI som lär sig av spel utan att vi skriver regler

Bra jobbat! 🎉
